# When is an A/B test winner real?
An analysis of 27,616 randomized headline tests from the Upworthy Research Archive (CC BY 4.0, [OSF](https://osf.io/jd64p/)).

**Question:** if a product team ships the headline that won its test, did it win, or got lucky? Every number below comes from the cells in this notebook. Put the two CSVs in a `data/` folder (see README) and run top to bottom.

The archive has an *exploratory* set (4,873 tests) and a *confirmatory* set (22,743 tests). I explore on the first and check what I found on the second, which was set aside as fresh tests.

In [ ]:
import os, re, numpy as np, pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
DATA = os.environ.get("UPWORTHY_DIR", "data")
rng = np.random.default_rng(42)
def load(name):
    d = pd.read_csv(f"{DATA}/upworthy-archive-{name}-packages-03.12.2020.csv", low_memory=False)
    d["created_at"] = pd.to_datetime(d["created_at"], format="mixed")
    d = d[["clickability_test_id","created_at","headline","impressions","clicks","test_week"]].copy()
    d["test"] = d.clickability_test_id
    return d
raw = {n: load(n) for n in ["exploratory","confirmatory"]}
for n,d in raw.items():
    print(n, f"{len(d):,} packages | {d.test.nunique():,} tests | {d.impressions.sum():,} impressions | {d.clicks.sum():,} clicks")

## 1. Was traffic split fairly?
A randomized test should show each headline to about the same number of people. I test that for every test. The archive notes a problem period (Jun 25 2013 to Jan 10 2014), and the check below confirms it.

In [ ]:
# In a randomized test every arm should get about the same traffic. Chi-square on impressions per test.
def prep(d):
    g = d.groupby("test")
    t = g.agg(start=("created_at","min"), arms=("clicks","size"), imp=("impressions","sum"), clk=("clicks","sum"))
    t["balance_p"] = g.apply(lambda x: stats.chisquare(x.impressions, [x.impressions.mean()]*len(x)).pvalue, include_groups=False)
    t["in_window"] = (t.start >= "2013-06-25") & (t.start < "2014-01-11")
    return t
T = {n: prep(d) for n,d in raw.items()}
for n,t in T.items():
    print(n, "| unbalanced (p<.01) inside Jun 25 2013-Jan 10 2014 window:", f"{(t.balance_p[t.in_window]<.01).mean():.1%}",
          f"of {t.in_window.sum():,} tests | outside window:", f"{(t.balance_p[~t.in_window]<.01).mean():.1%}", f"of {(~t.in_window).sum():,} tests")

## 2. Clean sample
Drop tests that started in the problem window. Tests where no headline got a click are dropped too, since there is nothing to compare.

In [ ]:
clean = {}
for n,d in raw.items():
    keep = T[n].index[~T[n].in_window]
    c = d[d.test.isin(keep)].copy()
    tot = c.groupby("test")[["impressions","clicks"]].transform("sum")
    c["ctr"] = c.clicks / c.impressions
    c["test_ctr"] = tot.clicks / tot.impressions
    c = c[c.test.map(c.groupby("test").clicks.sum()) > 0]
    clean[n] = c
    print(n, f"clean: {c.test.nunique():,} tests, {len(c):,} packages, {c.impressions.sum():,} impressions, {c.clicks.sum():,} clicks, overall CTR {c.clicks.sum()/c.impressions.sum():.3%}")

## 3. Do headlines matter at all?
For each test, a chi-square test asks whether the headlines' click rates differ more than chance allows.

In [ ]:
def homog_p(x):
    t = np.array([x.clicks.values, (x.impressions - x.clicks).values])
    return stats.chi2_contingency(t, correction=False)[1]
for n,c in clean.items():
    p = c.groupby("test").apply(homog_p, include_groups=False)
    print(n, f"tests where arms differ in CTR (p<.05): {(p<.05).mean():.1%}  (pure noise would give 5%)  | p<.001: {(p<.001).mean():.1%}")
    T[n].loc[p.index,"homog_p"] = p

## 4. What can a typical test detect?
A headline typically got about 3,100 impressions at about a 1.3% click rate.

In [ ]:
c = clean["exploratory"]
arm_imp = c.impressions.median(); base = c.clicks.sum()/c.impressions.sum()
se = np.sqrt(2*base*(1-base)/arm_imp)
mde_abs = (1.96+0.84)*se
print(f"median impressions per headline: {arm_imp:,.0f} | baseline CTR {base:.2%} | smallest lift a typical test detects (80% power, 5%): {mde_abs:.2%} points = {mde_abs/base:.0%} relative")

## 5. How big is the winner's curse?
In each test I take the headline with the highest click rate and compare it to the average of the others. Then I simulate the same test 200 times with every headline given the *same* true click rate and see what lift the top headline shows from luck alone.

In [ ]:
def top_lift(g):
    ctr = g.clicks/g.impressions
    i = ctr.values.argmax()
    others = np.delete(ctr.values, i)
    return ctr.values[i]/others.mean() - 1
def noise_lift(g, reps=200):
    p = g.clicks.sum()/g.impressions.sum(); n = g.impressions.values
    out = np.empty(reps)
    for r in range(reps):
        s = rng.binomial(n, p)/n
        i = s.argmax(); out[r] = s[i]/np.delete(s,i).mean() - 1 if np.delete(s,i).mean()>0 else np.nan
    return np.nanmean(out)
sample = {}
for n,c in clean.items():
    ids = c.test.unique()
    if n=="confirmatory": ids = rng.choice(ids, 4000, replace=False)
    s = c[c.test.isin(ids)]
    obs = s.groupby("test").apply(top_lift, include_groups=False)
    noi = s.groupby("test").apply(noise_lift, include_groups=False)
    sample[n] = pd.DataFrame({"obs":obs,"noise":noi,"arms":s.groupby("test").size()})
    r = sample[n]
    print(n, f"| median top-arm lift vs rest: {r.obs.median():.1%} | median lift pure noise alone would give: {r.noise.median():.1%} | share of tests where top arm beats its noise-only expectation: {(r.obs>r.noise).mean():.1%}")
r = sample["exploratory"]
print(r.groupby(pd.cut(r.arms,[1,2,3,4,5,6,20])).agg(tests=("obs","size"),observed=("obs","median"),noise_only=("noise","median")).round(3))

## 6. How much of the spread between headlines is real?
Observed differences are real differences plus sampling noise. Subtracting the expected noise gives the share that is real.

In [ ]:
# deviation of each headline's log CTR from its test average; sampling variance ~ 1/clicks
rows=[]
for n,c in clean.items():
    c = c[c.clicks>=5].copy()
    c["y"] = np.log(c.ctr/c.test_ctr)
    c["v"] = 1/c.clicks - 1/c.impressions
    tau2 = max(0, (c.y**2).mean() - c.v.mean())
    rows.append((n, c.y.var(), c.v.mean(), tau2, tau2/(tau2+c.v.mean())))
print(pd.DataFrame(rows, columns=["data","observed_var","sampling_var","true_var","share_real"]).round(4))

## 7. What do winning headlines have in common?
Train on the exploratory tests: compare each headline with the other headlines *in its own test*, weighted by clicks, with standard errors clustered by test. Then fit the same model on the confirmatory tests.

In [ ]:
def feats(h):
    h = str(h); l = h.lower()
    return dict(
        words = len(h.split()),
        question = int("?" in h),
        number = int(bool(re.search(r"\d", h))),
        starts_number = int(bool(re.match(r"\s*\d", h))),
        you = int(bool(re.search(r"\byou(r|'re|'ll|'ve)?\b", l))),
        this = int(bool(re.search(r"\b(this|these)\b", l))),
        emotion = int(bool(re.search(r"\b(amazing|incredible|heartbreaking|beautiful|powerful|shocking|awesome|terrifying|stunning|brilliant|hilarious|adorable|mind|blown|breaks|crying|tears)\b", l))),
        negative = int(bool(re.search(r"\b(worst|terrible|disgusting|never|no one|nobody|hate|stop|wrong|fail|dies|dead|racist|sexist)\b", l))),
        quote = int(bool(re.search(r"[\"“”']", h)))
    )
X = {}
for n,c in clean.items():
    F = pd.DataFrame([feats(h) for h in c.headline], index=c.index)
    d = pd.concat([c[["test","impressions","clicks","ctr","test_ctr"]], F], axis=1)
    d = d[d.clicks>=1].copy(); d["y"] = np.log(d.ctr/d.test_ctr)
    X[n] = d
cols = list(feats("x").keys())
def wls(d, cols):
    # within-test demeaned; weight by clicks; cluster-robust SE by test
    z = d[cols+["y"]].copy(); w = d.clicks.values.astype(float)
    key = d.test.values
    for k in cols+["y"]:
        z[k] = z[k].values - pd.Series(z[k].values*w).groupby(key).transform("sum").values/pd.Series(w).groupby(key).transform("sum").values
    A = z[cols].values; y = z["y"].values
    W = w[:,None]; XtWX = A.T@(W*A); beta = np.linalg.solve(XtWX, A.T@(w*y))
    e = y - A@beta; sc = (A*(w*e)[:,None])
    S = pd.DataFrame(sc).groupby(key).sum().values
    V = np.linalg.inv(XtWX) @ (S.T@S) @ np.linalg.inv(XtWX)
    return pd.DataFrame({"effect_pct":(np.exp(beta)-1)*100,"se":np.sqrt(np.diag(V)),"t":beta/np.sqrt(np.diag(V))}, index=cols), beta
res_e, b_e = wls(X["exploratory"], cols)
res_c, b_c = wls(X["confirmatory"], cols)
out = res_e.join(res_c, lsuffix="_explore", rsuffix="_confirm")[["effect_pct_explore","t_explore","effect_pct_confirm","t_confirm"]].round(2)
print(out)

## 8. Does the rule work on tests it has not seen?
Use the exploratory-trained rule to pick one headline per confirmatory test and compare with picking at random.

In [ ]:
def pick_eval(d, beta):
    s = d[cols].values@beta
    d = d.assign(score=s)
    out=[]
    for t,g in d.groupby("test"):
        if len(g)<2: continue
        pick = g.score.values.argmax(); ctr = g.ctr.values
        rest = np.delete(ctr,pick)
        out.append((ctr[pick] > rest.mean(), ctr[pick]/rest.mean()-1, ctr[pick] >= ctr.max()))
    o = pd.DataFrame(out, columns=["beats_avg","uplift","is_best"])
    return o
o = pick_eval(X["confirmatory"], b_e)
print(f"On {len(o):,} confirmatory tests, picking the headline the exploratory-trained rule likes best:")
print(f"  beats the test average of the other headlines: {o.beats_avg.mean():.1%} (coin flip would be ~50%)  | average uplift: {o.uplift.mean():.1%}")
# random baseline
rb = []
for t,g in X["confirmatory"].groupby("test"):
    if len(g)<2: continue
    i = rng.integers(len(g)); ctr=g.ctr.values; rb.append((ctr[i]>np.delete(ctr,i).mean(), ctr[i]/np.delete(ctr,i).mean()-1))
rb = pd.DataFrame(rb, columns=["beats_avg","uplift"])
print(f"  random pick: beats average {rb.beats_avg.mean():.1%}, average uplift {rb.uplift.mean():.1%}")

## 9. Charts and files for the dashboard

In [ ]:
os.makedirs("outputs", exist_ok=True)
# 1. winner's curse by number of arms
r = pd.concat([sample["exploratory"].assign(data="exploratory"), sample["confirmatory"].assign(data="confirmatory")])
r["arms_group"] = pd.cut(r.arms,[1,2,3,4,5,6,20],labels=["2","3","4","5","6","7+"])
wc = r.groupby(["data","arms_group"],observed=True).agg(tests=("obs","size"), observed_lift=("obs","median"), noise_only_lift=("noise","median")).reset_index()
wc.to_csv("outputs/winners_curse_by_arms.csv", index=False)
e = wc[wc.data=="exploratory"]
fig, ax = plt.subplots(figsize=(7,4))
x = np.arange(len(e)); ax.bar(x-0.2, e.observed_lift*100, 0.4, label="Observed top-headline lift")
ax.bar(x+0.2, e.noise_only_lift*100, 0.4, label="Lift pure noise would give")
ax.set_xticks(x); ax.set_xticklabels(e.arms_group); ax.set_xlabel("Headlines in the test"); ax.set_ylabel("Median lift vs the others (%)")
ax.set_title("More headlines, bigger apparent winner, mostly noise"); ax.legend(frameon=False); plt.tight_layout(); plt.savefig("outputs/winners_curse.png", dpi=160); plt.show()
# 2. feature effects, explore vs confirm
fe = out.reset_index().rename(columns={"index":"feature"}); fe.to_csv("outputs/feature_effects.csv", index=False)
fe2 = fe[fe.feature!="starts_number"].sort_values("effect_pct_confirm")
fig, ax = plt.subplots(figsize=(7,4)); y = np.arange(len(fe2))
ax.scatter(fe2.effect_pct_explore, y, label="Exploratory", s=40); ax.scatter(fe2.effect_pct_confirm, y, label="Confirmatory (held-out tests)", s=40, marker="D")
ax.axvline(0, color="grey", lw=.8); ax.set_yticks(y); ax.set_yticklabels(fe2.feature); ax.set_xlabel("Change in click rate vs same-test headlines (%)")
ax.set_title("Headline features: same pattern on held-out tests"); ax.legend(frameon=False); plt.tight_layout(); plt.savefig("outputs/feature_effects.png", dpi=160); plt.show()
# 3. balance by month
bal=[]
for n,t in T.items():
    m = t.assign(month=t.start.dt.to_period("Q").astype(str), bad=t.balance_p<.01).groupby("month").agg(tests=("bad","size"), unbalanced_share=("bad","mean")).reset_index(); m["data"]=n; bal.append(m)
pd.concat(bal).to_csv("outputs/balance_by_quarter.csv", index=False)
b = bal[0]; fig, ax = plt.subplots(figsize=(7,3.5)); ax.bar(b.month, b.unbalanced_share*100); ax.axhline(1, color="red", lw=.8, ls="--")
ax.set_ylabel("Tests with unbalanced traffic (%)"); ax.set_title("Traffic split was broken Jun 2013 to Jan 2014"); plt.xticks(rotation=60, fontsize=7); plt.tight_layout(); plt.savefig("outputs/balance.png", dpi=160); plt.show()
# kpis
k = dict(tests_raw=raw["exploratory"].test.nunique()+raw["confirmatory"].test.nunique(),
         tests_clean=sum(c.test.nunique() for c in clean.values()), top_lift_observed=float(sample["confirmatory"].obs.median()),
         top_lift_noise=float(sample["confirmatory"].noise.median()), arms_differ_share=float(T["confirmatory"].homog_p.dropna().lt(.05).mean()),
         mde_relative=float(mde_abs/base), pick_beats_avg=float(o.beats_avg.mean()), random_beats_avg=float(rb.beats_avg.mean()),
         pick_uplift=float(o.uplift.mean()), random_uplift=float(rb.uplift.mean()))
pd.Series(k).to_csv("outputs/kpis.csv", header=["value"]); print(pd.Series(k))